# Notebook 1 - Watershed Preparation and Hydrology

**Author:** Alex DeRosa  
**Date:** 2026-05-27  

**Toolset:** ArcGIS Pro, ArcPy, Spatial Analyst, and custom Python toolboxes  
**Project Focus:** Hydrologic preprocessing and watershed delineation for SWAT-style watershed analysis

---

#### Overview

This notebook prepares the core watershed framework for the Queen-Usquepaug study area. It processes source elevation and vector datasets, conditions the DEM for hydrologic routing, generates flow direction and flow accumulation rasters, extracts stream networks, and delineates watershed and subbasin boundaries.

The outputs from this notebook provide the spatial foundation for later SWAT-style land cover, soils, slope, summary, and HRU workflows.

---

#### Major Objectives

- Define project paths, geodatabases, environments, and shared constants
- Prepare and validate source spatial datasets
- Project and clip required vector inputs
- Condition the DEM for hydrologic routing
- Preserve major sink and pond features where appropriate
- Generate flow direction and flow accumulation rasters
- Create stream raster, stream link, and stream network outputs
- Delineate subbasin raster and polygon outputs
- Add dominant town attribution to subbasin polygons
- Confirm final outputs needed by Notebook 2

---

#### Workflow

**Task 0: Notebook Setup**  
Defines imports, project paths, geodatabases, output folders, coordinate systems, shared constants, input datasets, output datasets, and optional cleanup settings.

**Task 1: Prepare Study Area and Vector Inputs**  
Creates the study area boundary and prepares required vector datasets, including rivers, towns, soils, and land cover.

**Task 2: Prepare DEM**  
Extracts, projects, resamples, and validates the DEM for hydrologic processing.

**Task 3: Prepare Terrain Environment**  
Configures raster processing environments and prepares terrain datasets required for flow routing.

**Task 4: Condition DEM and Preserve Real Sinks**  
Hydrologically conditions the DEM, evaluates sink regions, preserves major pond and depression features, and creates the final hydro-conditioned DEM.

**Task 5: Create Stream Network**  
Generates flow direction and flow accumulation rasters, extracts streams using the selected accumulation threshold, and creates stream raster, stream link, and stream network outputs.

**Task 6: Delineate Subbasins**  
Creates subbasin raster and polygon outputs for the study area.

**Task 7: Add Dominant Town Attribution**  
Assigns dominant municipal attribution to each subbasin using overlap-based spatial attribution.

**Task 8: Final Handoff QA**  
Confirms that all Notebook 1 outputs required by Notebook 2 exist and are ready for downstream SWAT spatial input processing.

# TASK 0 - PROJECT SETUP

#### Step 1: Import Libraries and Custom Tools

In [ ]:
# Import Python Libraries

import os
import sys
import csv

import arcpy

arcpy.CheckOutExtension("Spatial")
from arcpy.sa import *

print("Python libraries imported.")
print("Spatial Analyst enabled.")

In [ ]:
# Import custom Python toolboxes
base_folder    = r"<BASE_ARCGIS_FOLDER>"
toolbox_folder = os.path.join(base_folder, "Toolboxes")

arcpy.ImportToolbox(os.path.join(toolbox_folder, "Field_Tools.pyt"), "field_tools")
arcpy.ImportToolbox(os.path.join(toolbox_folder, "Raster_Tools.pyt"), "raster_tools")
arcpy.ImportToolbox(os.path.join(toolbox_folder, "Vector_Tools.pyt"), "vector_tools")

print("Custom Python toolboxes imported.")
print("Field tools  : field_tools")
print("Raster tools : raster_tools")
print("Vector tools : vector_tools")

#### Step 2: Define Project Paths and Folders

In [ ]:
# Define project paths
project_name = "Queen_Usquepaug"
project_folder = os.path.join(base_folder, "Projects", project_name)

# Define geodatabases
project_gdb = os.path.join(project_folder, f"{project_name}.gdb")
temp_gdb    = os.path.join(project_folder, "Temp.gdb")

# Define folders
inputs_folder  = os.path.join(project_folder, "Inputs")
outputs_folder = os.path.join(project_folder, "Outputs")
tables_folder  = os.path.join(outputs_folder, "Tables")
charts_folder  = os.path.join(outputs_folder, "Charts")

# Define source data folders
source_data_folder = os.path.join(base_folder, "GIS_Data")
ri_data            = os.path.join(source_data_folder, "RI_State_Data")
lidar_data         = os.path.join(source_data_folder, "LIDAR")

print("Input data paths defined.")

In [ ]:
print("Project paths defined.")
print(f" Project folder : {project_folder}")

print("Geodatabases defined.")
print(f" Project GDB : {project_gdb}")
print(f" Temp GDB    : {temp_gdb}")

print("Folders defined.")
print(f" Inputs folder  : {inputs_folder}")
print(f" Tables folder : {tables_folder}")
print(f" Charts folder : {charts_folder}")

print("Source data folders defined.")
print(f" Source data : {source_data_folder}")
print(f" RI data     : {ri_data}")
print(f" LIDAR data  : {lidar_data}")

In [ ]:
# Create Project GDB and Temp GDB if it does not already exist

if not arcpy.Exists(project_gdb):
    arcpy.management.CreateFileGDB(out_folder_path=project_folder, out_name=f"{project_name}.gdb")
    print("Project GDB created.")

else:
    print("Project GDB already exists.")

# temp gdb

if not arcpy.Exists(temp_gdb):
    arcpy.management.CreateFileGDB(out_folder_path=project_folder, out_name="Temp.gdb")
    print("Temp GDB created.")

else:
    print("Temp GDB already exists.")

In [ ]:
# Create Project Output Folders

for folder in [outputs_folder, tables_folder, charts_folder]:
    os.makedirs(folder, exist_ok=True)
    
print("Output folders created or confirmed.")

#### Step 3: Configure ArcPy Environment

In [ ]:
# Configure workspace

arcpy.env.workspace = temp_gdb
arcpy.env.scratchWorkspace = temp_gdb

arcpy.env.overwriteOutput = True

print("Environment configured.")
print(f"Default workspace : {arcpy.env.workspace}")
print(f"Scratch workspace : {arcpy.env.scratchWorkspace}")
print(f"Important outputs : {project_gdb}")
print(f"Overwrite Enabled : {arcpy.env.overwriteOutput}")

In [ ]:
# Define Coordinate System

ri_stateplane_meters = arcpy.SpatialReference(32130)
arcpy.env.outputCoordinateSystem = ri_stateplane_meters

print("Spatial reference set.")
print(f"CRS : {ri_stateplane_meters.name}")
print(f"WKID: {ri_stateplane_meters.factoryCode}")

#### Step 4: Define Shared Constants

In [ ]:
# Define Shared Project Constants

study_area_name = "Queen_Usquepaug"
stream_threshold = 29000
swat_cell_size = 10

# Define Unit Conversion Constants
sq_m_to_acres = 0.000247105

print("Project constants defined.")
print(f" Study area       : {study_area_name}")
print(f" Stream threshold : {stream_threshold}")
print(f" SWAT cell size   : {swat_cell_size}")

print("Unit conversion constants defined.")
print(f" Square meters to acres : {sq_m_to_acres}")

#### Step 5: Define Notebook Inputs

In [ ]:
# Define Source Data Paths

# National data
huc12 = os.path.join(source_data_folder, "National_Data", "USGS", "NHDPLUS_H_0109_HU4_GDB.gdb", "WBD", "WBDHU12")

# Rhode Island source datasets
town_boundaries = os.path.join(ri_data, "03_Boundaries", "Towns.shp")
rivers = os.path.join(ri_data, "02_Hydrography", "WoodPaw_WildScenic_Rivers", "WoodPaw_WildScenic_Rivers.shp")
soils  = os.path.join(ri_data, "05_Environmental", "Soils", "SOIL_Soils25_spf.shp")
lulc   = os.path.join(ri_data, "01_LandCover", "RILC_2011", "rilc11d.shp")

# DEM source dataset
dem_original = os.path.join(inputs_folder, "qu_mosaic")

print("Source data paths defined.")

In [ ]:
# Validate Required Source Inputs

required_inputs = {
    "HUC12": huc12,
    "Town Boundaries": town_boundaries,
    "Rivers": rivers,
    "Soils": soils,
    "LULC": lulc,
    "DEM": dem_original}

missing_inputs = []

print("Input dataset QA:")

for name, path in required_inputs.items():
    exists = arcpy.Exists(path)
    print(f"{name:<18}: {'FOUND' if exists else 'MISSING'}")

    if not exists:
        missing_inputs.append(name)

if missing_inputs:
    raise FileNotFoundError(f"Missing required inputs: {missing_inputs}")

print("")
print("Input dataset QA: PASS")

#### Step 6: Define Notebook 1 Outputs

In [ ]:
# Define Study Area and Prepared Vector Outputs

study_area = os.path.join(project_gdb, f"{study_area_name}_StudyArea")

rivers_projected = os.path.join(project_gdb, "Rivers_Projected")
soils_projected  = os.path.join(project_gdb, "Soils_Projected")
lulc_projected   = os.path.join(project_gdb, "LULC_Projected")
towns_projected  = os.path.join(project_gdb, "Towns_Projected")

print("Study area and prepared vector outputs defined.")
print(f"Study area output : {study_area}")

In [ ]:
# Define DEM and hydrology outputs

dem_hydro    = os.path.join(project_gdb, "DEM_HydroConditioned")
flow_dir     = os.path.join(project_gdb, "Flow_Direction")
flow_acc     = os.path.join(project_gdb, "Flow_Accumulation")
flow_acc_log = os.path.join(project_gdb, "Flow_Accumulation_Log")

print("DEM and hydrology outputs defined.")

In [ ]:
# Define stream and subbasin outputs
stream_raster  = os.path.join(project_gdb, f"StreamRaster_T{stream_threshold}")
stream_links   = os.path.join(project_gdb, f"StreamLinks_T{stream_threshold}")
stream_network = os.path.join(project_gdb, f"StreamNetwork_T{stream_threshold}")

subbasin_raster   = os.path.join(project_gdb, f"SubbasinRaster_T{stream_threshold}")
subbasin_polygons = os.path.join(project_gdb, f"SubbasinPolygons_T{stream_threshold}")

print("Stream and subbasin outputs defined.")
print(f" Unique stream segment IDs : {stream_links}")
print(f" Mapped stream polylines   : {stream_network}")
print(f" Subbasin polygons         : {subbasin_polygons}")

#### Step 7: Optional Clear Temp GDB

In [ ]:
#  Clear Temp GDB

clear_temp_gdb = False

if clear_temp_gdb:

    arcpy.env.workspace = temp_gdb

    datasets = (
        (arcpy.ListRasters() or []) +
        (arcpy.ListFeatureClasses() or []) +
        (arcpy.ListTables() or []) +
        (arcpy.ListDatasets() or []))

    print("Clearing Temp GDB:")

    for dataset in datasets:
        try:
            arcpy.management.Delete(dataset)
            print(f"Deleted : {dataset}")
        except Exception as e:
            print(f"Skipped : {dataset} ({e})")
            
    print("Temp GDB cleared.")

else:
    print("Temp GDB cleanup skipped.")

# TASK 1 - PREPARE STUDY AREA AND VECTOR INPUTS

#### Step 1: Define Task Outputs

In [ ]:
# Define temporary vector outputs

study_area_selected = os.path.join(temp_gdb, "StudyArea_Selected")

rivers_clipped = os.path.join(temp_gdb, "Rivers_Clipped")
soils_clipped  = os.path.join(temp_gdb, "Soils_Clipped")
lulc_clipped   = os.path.join(temp_gdb, "LULC_Clipped")
towns_clipped  = os.path.join(temp_gdb, "Towns_Clipped")

print("Temporary vector outputs defined.")

#### Step 2: Inspect HUC12 Fields and Define Selection Query

In [ ]:
# Inspect HUC12 Fields - confirm field names before selecting watershed

for field in arcpy.ListFields(huc12):
    print(field.name)

In [ ]:
# Define Study Area Selection Query

where_clause    = "Name = 'Usquepaug River'"

print(f" Study area name : {study_area_name}")
print(f" Where clause    : {where_clause}")

#### Step 3: Select and Export Study Area

In [ ]:
# Create HUC12 feature layer

huc12_layer = "huc12_layer"

arcpy.management.MakeFeatureLayer(in_features=huc12, out_layer=huc12_layer)

print("HUC12 feature layer created.")

In [ ]:
# Select study area watershed

arcpy.management.SelectLayerByAttribute(in_layer_or_view=huc12_layer, selection_type="NEW_SELECTION", where_clause=where_clause)

print(f"Selected features.")

In [ ]:
# Check Selected Feature Count

selected_count = int(arcpy.management.GetCount(huc12_layer)[0])

print(f"Selected features: {selected_count}")

if selected_count == 0:
    raise ValueError("No HUC12 selected. Check field name or watershed name.")

if selected_count > 1:
    raise ValueError("More than one HUC12 selected. Check selection query.")

In [ ]:
# Export selected study area to Temp GDB

if arcpy.Exists(study_area_selected):
    arcpy.management.Delete(study_area_selected)

arcpy.management.CopyFeatures(in_features=huc12_layer, out_feature_class=study_area_selected)

print("Selected study area exported.")

In [ ]:
# Clear selection and delete temporary layer

arcpy.management.SelectLayerByAttribute(in_layer_or_view=huc12_layer, selection_type="CLEAR_SELECTION")

arcpy.management.Delete(in_data=huc12_layer)

print("Temporary HUC12 layer cleaned up.")

In [ ]:
# Project Selected Study Area

if arcpy.Exists(study_area):
    arcpy.management.Delete(study_area)

arcpy.management.Project(in_dataset=study_area_selected, out_dataset=study_area, out_coor_system=ri_stateplane_meters)

print("Study area projected.")

In [ ]:
# Confirm study area was created correctly

if not arcpy.Exists(study_area):
    raise FileNotFoundError(f"Study area missing: {study_area}")

study_area_count = int(arcpy.management.GetCount(study_area)[0])

desc = arcpy.Describe(study_area)

print("Final study area QA:")
print(f"Count : {study_area_count}")
print(f"CRS   : {desc.spatialReference.name}")
print(f"Path  : {study_area}")

print(
    f"QA Check : "
    f"{'PASS' if study_area_count == 1 else 'FAIL'}")

#### Step 4: Prepare Vector Inputs

In [ ]:
# Check source coordinate systems

for name, path in required_inputs.items():
    sr = arcpy.Describe(path).spatialReference

    print(f"{name:<18}: {sr.name}")

In [ ]:
# Clip and project rivers

arcpy.vector_tools.Clip_Project_Dataset(
    input_features=rivers,
    clip_features=study_area,
    output_features=rivers_projected,
    spatial_reference_wkid=ri_stateplane_meters.factoryCode,
    repair_geometry=True,
    temp_workspace=temp_gdb)

In [ ]:
# Clip and project land cover

arcpy.vector_tools.Clip_Project_Dataset(
    input_features=lulc,
    clip_features=study_area,
    output_features=lulc_projected,
    spatial_reference_wkid=ri_stateplane_meters.factoryCode,
    repair_geometry=True,
    temp_workspace=temp_gdb)

In [ ]:
# Clip and project soils

arcpy.vector_tools.Clip_Project_Dataset(
    input_features=soils,
    clip_features=study_area,
    output_features=soils_projected,
    spatial_reference_wkid=ri_stateplane_meters.factoryCode,
    repair_geometry=True,
    temp_workspace=temp_gdb)

In [ ]:
# Clip and project towns

arcpy.vector_tools.Clip_Project_Dataset(
    input_features=town_boundaries,
    clip_features=study_area,
    output_features=towns_projected,
    spatial_reference_wkid=ri_stateplane_meters.factoryCode,
    repair_geometry=True,
    temp_workspace=temp_gdb)

#### Step 5: Prepared Vector QA Checks

In [ ]:
# Prepared vector QA

vector_outputs = {
    "Study Area": study_area,
    "Rivers": rivers_projected,
    "Soils": soils_projected,
    "LULC": lulc_projected,
    "Towns": towns_projected}

qa_pass = True

print("Prepared vector QA:")

for name, path in vector_outputs.items():
    if not arcpy.Exists(path):

        print(f"{name:<14}: MISSING")
        qa_pass = False
        continue

    count = int(arcpy.management.GetCount(path)[0])

    desc = arcpy.Describe(path)

    crs_ok = (desc.spatialReference.factoryCode == ri_stateplane_meters.factoryCode)

    print(
        f"{name:<14}: "
        f"{count:<8} features | "
        f"CRS: {'PASS' if crs_ok else 'FAIL'}")

    if count == 0 or not crs_ok:
        qa_pass = False

print(f"\nVector QA: {'PASS' if qa_pass else 'FAIL'}")

# TASK 2 - DEM PREPARATION

#### Step 1: Define DEM Preparation Outputs

In [ ]:
# Define temporary DEM preparation outputs

dem_masked    = os.path.join(temp_gdb, "DEM_Masked")
dem_projected = os.path.join(temp_gdb, f"DEM_{swat_cell_size}m")

print("DEM preparation outputs defined.")
print(f" Masked    : {dem_masked}")
print(f" Projected : {dem_projected}")

In [ ]:
# Check current DEM

dem = dem_original

print("Current DEM properties.")
print(f" Original DEM   : {dem_original}")
print(f" SWAT cell size : {swat_cell_size} meters")
print(f" Current DEM    : {dem}")

#### Step 2: Create Working DEM

In [ ]:
# Extract and project DEM

arcpy.raster_tools.Extract_Project_Raster_Dataset(
    input_raster=dem,
    study_area=study_area,
    output_raster=dem_projected,
    target_crs_wkid=ri_stateplane_meters.factoryCode,
    output_cell_size=swat_cell_size,
    resampling_type="BILINEAR")

dem = dem_projected

print("DEM resampled to study area.")
print(f"Current DEM: {dem}")

In [ ]:
# Calculate DEM statistics

arcpy.management.CalculateStatistics(dem)

print("DEM statistics calculated.")

In [ ]:
# QA - Working DEM Properties

if not arcpy.Exists(dem):
    raise FileNotFoundError(f"Active DEM missing: {dem}")

dem_description = arcpy.Describe(dem)

cell_width   = dem_description.meanCellWidth
cell_height  = dem_description.meanCellHeight
crs_ok       = (dem_description.spatialReference.factoryCode == ri_stateplane_meters.factoryCode)
cell_size_ok = (
    round(cell_width, 6) == swat_cell_size and
    round(cell_height, 6) == swat_cell_size)

print("Working DEM QA:")
print(f"Active DEM      : {dem}")
print(f"Projection      : {dem_description.spatialReference.name}")
print(f"Cell width      : {cell_width} m")
print(f"Cell height     : {cell_height} m")
print(f"CRS Check       : {'PASS' if crs_ok else 'FAIL'}")
print(f"Cell Size Check : {'PASS' if cell_size_ok else 'FAIL'}")

print(
    f"\nDEM QA: "
    f"{'PASS' if crs_ok and cell_size_ok else 'FAIL'}")

# TASK 3 - TERRAIN PREPARATION

#### Step 1: Define Terrain Processing Outputs

In [ ]:
# Define terrain preparation outputs

sink_group_summary_csv = os.path.join(tables_folder, "Sink_Group_Size_Summary.csv")

flow_dir_raw     = os.path.join(temp_gdb, "FlowDir_Raw")
flow_acc_raw     = os.path.join(temp_gdb, "FlowAcc_Raw")
flow_acc_raw_log = os.path.join(temp_gdb, "FlowAcc_Raw_Log")

sinks            = os.path.join(temp_gdb, "Sinks")
sink_groups      = os.path.join(temp_gdb, "Sink_Groups")
ponds            = os.path.join(temp_gdb, "Real_Sinks")

print("Terrain preparation outputs defined.")

In [ ]:
# Define sink interpretation parameters

# Minimum sink size to preserve as a real depression
# 80 cells ≈ 8,000 sq m ≈ 2 acres at 10 m resolution

real_sink_min_cells = 80

# Small elevation offset used when restoring preserved sinks
# 0.01 m = 1 cm

sink_restore_depth = 0.01

print(f"Real sink minimum size : {real_sink_min_cells} cells")
print(f"Sink restore depth     : {sink_restore_depth} m")
print(f"Current DEM            : {dem}")

#### Step 2: Generate Raw Hydrology Rasters

In [ ]:
# Create raw D8 flow direction raster

if arcpy.Exists(flow_dir_raw):
    arcpy.management.Delete(flow_dir_raw)

flow_dir_raw_result = FlowDirection(
    in_surface_raster=dem,
    force_flow="NORMAL",
    flow_direction_type="D8")

flow_dir_raw_result.save(flow_dir_raw)

print("Raw D8 flow direction raster created.")

In [ ]:
# Create raw flow accumulation raster

if arcpy.Exists(flow_acc_raw):
    arcpy.management.Delete(flow_acc_raw)

flow_acc_raw_result = FlowAccumulation(in_flow_direction_raster=flow_dir_raw)

flow_acc_raw_result.save(flow_acc_raw)

print("Raw flow accumulation raster created.")

In [ ]:
# Create log-transformed raw flow accumulation raster

if arcpy.Exists(flow_acc_raw_log):
    arcpy.management.Delete(flow_acc_raw_log)

flow_acc_raw_log_result = Log10(Raster(flow_acc_raw) + 1)

flow_acc_raw_log_result.save(flow_acc_raw_log)

print("Raw log flow accumulation raster created.")

#### Step 3: Identify Sink Regions

In [ ]:
# Create sink raster

if arcpy.Exists(sinks):
    arcpy.management.Delete(sinks)

sink_result = Sink(in_flow_direction_raster=flow_dir_raw)

sink_result.save(sinks)

print("Sink raster created.")

In [ ]:
# Group connected sink regions

if arcpy.Exists(sink_groups):
    arcpy.management.Delete(sink_groups)

sink_group_result = RegionGroup(in_raster=sinks, number_neighbors="EIGHT", zone_connectivity="WITHIN", add_link="ADD_LINK")

sink_group_result.save(sink_groups)

print("Sink regions grouped.")

In [ ]:
# Build Sink Group Attribute Table - required for sink group size filtering using COUNT.

arcpy.management.BuildRasterAttributeTable(in_raster=sink_groups, overwrite="Overwrite")

print("Sink group attribute table built.")

#### Step 4: Summarize Sink Group Sizes

In [ ]:
# Summarize sink group sizes

sink_counts = []

with arcpy.da.SearchCursor(sink_groups, ["Value", "Count"]) as cursor:
    for value, count in cursor:
        area_sq_m = (count * swat_cell_size * swat_cell_size)
        area_acres = area_sq_m * sq_m_to_acres

        sink_counts.append((value, count, area_sq_m, area_acres))

sink_counts = sorted(
    sink_counts,
    key=lambda x: x[1],
    reverse=True)

print("Sink groups summarized.")
print(f"Sink groups found: {len(sink_counts)}")

In [ ]:
# Preview largest sink groups

print("Largest sink groups:")

for value, count, area_sq_m, area_acres in sink_counts[:20]:

    print(
        f"Group {value}: "
        f"{count} cells | "
        f"{area_sq_m:,.0f} sq m | "
        f"{area_acres:,.2f} acres")

In [ ]:
# Export Sink Group Summary to CSV

if os.path.exists(sink_group_summary_csv):
    os.remove(sink_group_summary_csv)
    print("Existing sink group summary deleted.")

with open(sink_group_summary_csv, "w", newline="") as csvfile:

    writer = csv.writer(csvfile)

    writer.writerow(["Sink_Group_Value", "Cell_Count", "Area_sq_m", "Area_acres"])

    for row in sink_counts:
        writer.writerow(row)

print("Sink group summary exported.")
print(f"Output: {sink_group_summary_csv}")

# TASK 4 - CONDITION DEM & PRESERVE REAL SINKS

#### Step 1: Define Terrain Conditioning Outputs

In [ ]:
# Define temporary terrain conditioning outputs

ponds0_else1     = os.path.join(temp_gdb, "Ponds0_Else1")
ponds_mask       = os.path.join(temp_gdb, "Ponds_Mask")

filled_dem          = os.path.join(temp_gdb, "DEM_Filled_AllSinks")
sink_restore_raster = os.path.join(temp_gdb, "Sink_Restore_Raster")

print("Terrain conditioning outputs defined.")

#### Step 2: Preserve Large Sink Regions

In [ ]:
# Extract large sink regions to preserve. Small sink groups become NoData; preserved sink groups become 1

if arcpy.Exists(ponds):
    arcpy.management.Delete(ponds)

ponds_result = SetNull(in_conditional_raster=sink_groups, in_false_raster_or_constant=1, where_clause=f"COUNT < {real_sink_min_cells}")

ponds_result.save(ponds)

print("Large sink regions extracted.")

In [ ]:
# Create inverse sink raster. Preserved sinks = 0; all other cells = 1

if arcpy.Exists(ponds0_else1):
    arcpy.management.Delete(ponds0_else1)

ponds0_else1_result = IsNull(in_raster=ponds)

ponds0_else1_result.save(ponds0_else1)

print("0/1 pond raster created.")

In [ ]:
# Create preserved sink mask. Preserved sinks = 1; all other cells = 0

if arcpy.Exists(ponds_mask):
    arcpy.management.Delete(ponds_mask)

ponds_mask_result = EqualTo(in_raster_or_constant1=ponds0_else1, in_raster_or_constant2=0)

ponds_mask_result.save(ponds_mask)

print("Preserved sink mask created.")

In [ ]:
# Confirm pond mask has cells with value 1

arcpy.management.BuildRasterAttributeTable(in_raster=ponds_mask, overwrite="Overwrite")

with arcpy.da.SearchCursor(ponds_mask, ["Value", "Count"]) as cursor:
    for value, count in cursor:
        area_sq_m = count * swat_cell_size * swat_cell_size
        area_acres = area_sq_m * sq_m_to_acres
        print(f"Value {value}: {count:,.0f} cells | {area_acres:,.2f} acres")

#### Step 3: Fill DEM and Restore Preserved Sinks

In [ ]:
# Fill all sinks in DEM

if arcpy.Exists(filled_dem):
    arcpy.management.Delete(filled_dem)

filled_dem_result = Fill(in_surface_raster=dem)
filled_dem_result.save(filled_dem)

dem = filled_dem

print("Filled DEM created.")
print(f"Current DEM: {dem}")

In [ ]:
# Create sink restore raster

if arcpy.Exists(sink_restore_raster):
    arcpy.management.Delete(sink_restore_raster)

sink_restore_result = Times(in_raster_or_constant1=ponds_mask, in_raster_or_constant2=sink_restore_depth)

sink_restore_result.save(sink_restore_raster)

print("Sink restore raster created.")

In [ ]:
# Create hydro-conditioned DEM

if arcpy.Exists(dem_hydro):
    arcpy.management.Delete(dem_hydro)
    print("Existing hydro-conditioned DEM deleted.")

dem_hydro_result = Minus(in_raster_or_constant1=dem, in_raster_or_constant2=sink_restore_raster)
dem_hydro_result.save(dem_hydro)

dem = dem_hydro

print("Hydro-conditioned DEM created.")
print(f"Current DEM: {dem}")

In [ ]:
# Set raster environment to hydro-conditioned DEM

arcpy.env.snapRaster = dem
arcpy.env.cellSize   = dem
arcpy.env.extent     = dem
arcpy.env.mask       = study_area

desc = arcpy.Describe(dem)

print("Raster environment updated to hydro DEM.")
print(f"DEM         : {dem}")
print(f"Snap Raster : {arcpy.env.snapRaster}")
print(f"Cell Size   : {desc.meanCellWidth}")
print(f"Mask        : {arcpy.env.mask}")

#### Step 4: Terrain Conditioning QA

In [ ]:
# Terrain conditioning output QA

terrain_conditioning_outputs = {
    "Sink Group Summary CSV": sink_group_summary_csv,
    "Filled DEM": filled_dem,
    "Sink Restore Raster": sink_restore_raster,
    "Hydro DEM": dem_hydro}

qa_pass = True

print("Terrain conditioning output QA:")

for name, path in terrain_conditioning_outputs.items():

    exists = (
        os.path.exists(path)
        if str(path).lower().endswith(".csv")
        else arcpy.Exists(path))

    print(f"{name:<24}: {'OK' if exists else 'MISSING'}")

    if not exists:
        qa_pass = False

print(f"\nOutput QA: {'PASS' if qa_pass else 'FAIL'}")

In [ ]:
# Hydro-conditioned DEM properties QA

if not arcpy.Exists(dem_hydro):
    raise FileNotFoundError(f"Hydro-conditioned DEM missing: {dem_hydro}")

desc = arcpy.Describe(dem_hydro)

crs_ok = (desc.spatialReference.factoryCode == ri_stateplane_meters.factoryCode)

cell_size_ok = (round(desc.meanCellWidth, 6) == swat_cell_size and round(desc.meanCellHeight, 6) == swat_cell_size)

print("Hydro-conditioned DEM QA:")
print(f"DEM              : {dem}")
print(f"CRS              : {desc.spatialReference.name}")
print(f"Cell Width       : {desc.meanCellWidth}")
print(f"Cell Height      : {desc.meanCellHeight}")
print(f"CRS Check        : {'PASS' if crs_ok else 'FAIL'}")
print(f"Cell Size Check  : {'PASS' if cell_size_ok else 'FAIL'}")
print(f"\nDEM QA         : {'PASS' if crs_ok and cell_size_ok else 'FAIL'}")

In [ ]:
# DEM conditioning difference QA
# Difference = hydro-conditioned DEM minus fully filled DEM

dem_difference = Minus(in_raster_or_constant1=dem_hydro, in_raster_or_constant2=filled_dem)

min_difference = arcpy.management.GetRasterProperties(dem_difference, "MINIMUM")[0]

max_difference = arcpy.management.GetRasterProperties(dem_difference, "MAXIMUM")[0]

mean_difference = arcpy.management.GetRasterProperties(dem_difference, "MEAN")[0]

print("DEM conditioning difference QA:")
print(f"Minimum change : {min_difference}")
print(f"Maximum change : {max_difference}")
print(f"Mean change    : {mean_difference}")

# TASK 5 - CREATE STREAM NETWORK

#### Step 1: Define Stream Network Outputs

In [ ]:
# Define Stream Outputs

print("Temporary stream raster output defined.")
print(f"Stream raster: {stream_raster}")

#### Step 2: Create Final Flow-Routing Rasters

In [ ]:
# Create final flow direction raster

if arcpy.Exists(flow_dir):
    arcpy.management.Delete(flow_dir)

flow_dir_result = FlowDirection(in_surface_raster=dem, force_flow="NORMAL", flow_direction_type="D8")

flow_dir_result.save(flow_dir)

print("Final flow direction raster created.")

In [ ]:
# Create final flow accumulation raster. Calculate upstream contributing area for each cell

if arcpy.Exists(flow_acc):
    arcpy.management.Delete(flow_acc)

flow_acc_result = FlowAccumulation(in_flow_direction_raster=flow_dir, data_type="FLOAT")

flow_acc_result.save(flow_acc)

print("Final flow accumulation raster created.")

In [ ]:
# Create log-transformed final flow accumulation raster

if arcpy.Exists(flow_acc_log):
    arcpy.management.Delete(flow_acc_log)

flow_acc_log_result = Ln(in_raster_or_constant=Raster(flow_acc) + 1)

flow_acc_log_result.save(flow_acc_log)

print("Final log flow accumulation raster created.")

In [ ]:
# Calculate flow accumulation display statistics

arcpy.management.CalculateStatistics(in_raster_dataset=flow_acc_log)

print("Flow log accumulation statistics calculated.")

#### Step 3: Create Stream Network

In [ ]:
# Set raster processing environment

arcpy.env.snapRaster = flow_dir
arcpy.env.cellSize   = swat_cell_size
arcpy.env.extent     = study_area
arcpy.env.mask       = study_area

print("Raster environment set for stream extraction.")

In [ ]:
# Create stream raster, stream links, and stream network

print("Creating stream raster, stream links, and stream network...")

arcpy.raster_tools.Stream_Network_From_Flow_Accumulation(
    flow_direction=flow_dir,
    flow_accumulation=flow_acc,
    stream_threshold=stream_threshold,
    output_stream_raster=stream_raster,
    output_stream_links=stream_links,
    output_stream_network=stream_network)

In [ ]:
# Add stream length metrics (meters)

arcpy.field_tools.Add_Standard_Metric_Field(
    input_features=stream_network,
    metric_type="Length Meters",
    field_name="Stream_Length_m",
    field_alias="Stream Length (m)")

In [ ]:
# Add stream length metrics (kilometers)
arcpy.field_tools.Add_Standard_Metric_Field(
    input_features=stream_network,
    metric_type="Length Kilometers",
    field_name="Stream_Length_km",
    field_alias="Stream Length (km)")

#### Step 4: Stream Network QA

In [ ]:
# Count stream outputs

stream_count = int(arcpy.management.GetCount(stream_network)[0])

stream_link_count = len([row[0] for row in arcpy.da.SearchCursor(stream_links, ["Value"])])

print("Stream Network QA:")
print(f"Stream threshold : {stream_threshold}")
print(f"Stream features  : {stream_count:,}")
print(f"Stream links     : {stream_link_count:,}")

In [ ]:
# Check stream network outputs

stream_outputs = {
    "Flow Direction": flow_dir,
    "Flow Accumulation": flow_acc,
    "Flow Accumulation Log": flow_acc_log,
    "Stream Raster": stream_raster,
    "Stream Links": stream_links,
    "Stream Network": stream_network}

qa_pass = True

print("Stream network output QA:")

for name, path in stream_outputs.items():

    exists = arcpy.Exists(path)

    print(f"{name:<24}: {'OK' if exists else 'MISSING'}")

    if not exists:
        qa_pass = False

print(f"\nStream Network QA: {'PASS' if qa_pass else 'FAIL'}")

# TASK 6- DELINEATE SUBBASINS

#### Step 1: Define Subbasin Outputs

In [ ]:
# Define subbasin outputs

subbasin_raster_raw = os.path.join(temp_gdb, f"SubbasinRaster_Raw_T{stream_threshold}")

print("Subbasin outputs defined.")
print(f"Raw subbasin raster   : {subbasin_raster_raw}")
print(f"Final subbasin raster : {subbasin_raster}")
print(f"Subbasin polygons     : {subbasin_polygons}")

#### Step 2: Delineate Subbasins

In [ ]:
# Set raster environment to hydro-conditioned DEM

arcpy.env.snapRaster = dem
arcpy.env.cellSize   = dem
arcpy.env.extent     = dem
arcpy.env.mask       = study_area

desc = arcpy.Describe(dem)

print("Raster environment updated to hydro DEM.")
print(f"DEM         : {dem}")
print(f"Snap Raster : {arcpy.env.snapRaster}")
print(f"Cell Size   : {desc.meanCellWidth}")
print(f"Mask        : {arcpy.env.mask}")

In [ ]:
# Create subbasin raster from stream links

if arcpy.Exists(subbasin_raster):
    arcpy.management.Delete(subbasin_raster)

subbasin_raster_result = Watershed(in_flow_direction_raster=flow_dir, in_pour_point_data=stream_links)
subbasin_raster_result.save(subbasin_raster)

print("Subbasin raster created.")
print(subbasin_raster)

In [ ]:
# Build subbasin raster attribute table

arcpy.management.BuildRasterAttributeTable(in_raster=subbasin_raster, overwrite="Overwrite")

print("Subbasin raster attribute table built.")

In [ ]:
# Count unique subbasins

subbasin_ids = set()

with arcpy.da.SearchCursor(subbasin_raster, ["Value"]) as cursor:
    for row in cursor:
        subbasin_ids.add(row[0])

raw_subbasin_count = len(subbasin_ids)

print("Raw subbasin QA:")
print(f"Unique subbasins: {raw_subbasin_count:,}")

#### Step 3: Assign Unrepresented Watershed Cells

In [ ]:

subbasin_allocation = os.path.join(temp_gdb, f"SubbasinAllocation_T{stream_threshold}")
subbasin_raster_completed = os.path.join(temp_gdb, f"SubbasinRaster_Completed_T{stream_threshold}")

In [ ]:
# Assign only NoData cells to nearest subbasin

for raster in [subbasin_allocation, subbasin_raster_completed]:
    if arcpy.Exists(raster):
        arcpy.management.Delete(raster)

subbasin_allocation_result = EucAllocation(in_source_data=subbasin_raster, source_field="Value")
subbasin_allocation_result.save(subbasin_allocation)

subbasin_complete_result = Con(IsNull(subbasin_raster), subbasin_allocation, subbasin_raster)
subbasin_complete_result.save(subbasin_raster_completed)

print("Only NoData cells assigned to nearest subbasin.")
print(subbasin_raster_completed)

In [ ]:
# Replace final subbasin raster with completed raster

if arcpy.Exists(subbasin_raster):
    arcpy.management.Delete(subbasin_raster)

arcpy.management.CopyRaster(in_raster=subbasin_raster_completed, out_rasterdataset=subbasin_raster)

print("Final completed subbasin raster saved.")
print(subbasin_raster)

In [ ]:
# Rebuild raster attribute table

arcpy.management.BuildRasterAttributeTable(in_raster=subbasin_raster, overwrite="Overwrite")

print("Updated raster attribute table built.")

In [ ]:
# Count completed subbasins

subbasin_ids = set()

with arcpy.da.SearchCursor(subbasin_raster, ["Value"]) as cursor:
    for row in cursor:
        subbasin_ids.add(row[0])

completed_subbasin_count = len(subbasin_ids)

print("Completed subbasin QA:")
print(f"Unique subbasins: {completed_subbasin_count:,}")

#### Step 4: Convert Subbasins to Polygons

In [ ]:
# Convert subbasin raster to polygons

if arcpy.Exists(subbasin_polygons):
    arcpy.management.Delete(subbasin_polygons)

arcpy.conversion.RasterToPolygon(in_raster=subbasin_raster, out_polygon_features=subbasin_polygons, simplify="NO_SIMPLIFY", raster_field="Value")

print("Subbasin polygons created.")

In [ ]:
# Add Subbasin Area Field (m²)

arcpy.field_tools.Add_Standard_Metric_Field(
    input_features=subbasin_polygons,
    metric_type="Area Square Meters",
    field_name="Subbasin_Area_sq_m",
    field_alias="Subbasin Area (m²)")

In [ ]:
# Add Subbasin Area Field (acres)

arcpy.field_tools.Add_Standard_Metric_Field(
    input_features=subbasin_polygons,
    metric_type="Area Acres",
    field_name="Subbasin_Area_acres",
    field_alias="Subbasin Area (acres)")

In [ ]:
# Calculate subbasin statistics

polygon_feature_count = int(arcpy.management.GetCount(subbasin_polygons)[0])

subbasin_polygon_area_sq_m = 0
unique_subbasins = set()

with arcpy.da.SearchCursor(
    subbasin_polygons,
    ["gridcode", "Subbasin_Area_sq_m"]) as cursor:

    for gridcode, area in cursor:
        unique_subbasins.add(gridcode)
        subbasin_polygon_area_sq_m += area

subbasin_count = len(unique_subbasins)

print("Subbasin polygon statistics calculated.")
print(f"Polygon features      : {polygon_feature_count:,}")
print(f"Unique subbasins      : {subbasin_count:,}")
print(f"Total subbasin area   : {subbasin_polygon_area_sq_m:,.0f} m²")

# TASK 7 - ADD DOMINANT TOWN ATTRIBUTION

#### Step 1: Define Town Attribution Outputs

In [ ]:
# Define temporary town attribution outputs

subbasin_town_intersect = os.path.join(temp_gdb, "SubbasinTownIntersect")
subbasin_town_stats     = os.path.join(temp_gdb, "SubbasinTownStats")
subbasin_town_dominant  = os.path.join(temp_gdb, "SubbasinTownDominant")

In [ ]:
# Check projected town fields

for field in arcpy.ListFields(towns_projected):
    print(field.name)

#### Step 2: Intersect Subbasins with Towns

In [ ]:
# Add Dominant Town Attribution

arcpy.vector_tools.Dominant_Overlap_Attribution(
    base_features=subbasin_polygons,
    base_id_field="gridcode",
    overlay_features=towns_projected,
    overlay_attribute_field="NAME",
    output_field_name="Dominant_Town",
    temp_workspace=temp_gdb)

In [ ]:
# Alter dominant town field name

arcpy.management.AlterField(in_table=subbasin_polygons, field="Dominant_Town", new_field_alias="Dominant Town")

print("Dominant town attribution complete.")

#### Step 3: Dominant Town QA

In [ ]:
# Dominant town field QA

town_fields = [field.name for field in arcpy.ListFields(subbasin_polygons)]

print("Subbasin fields containing town/name:")

for field in town_fields:

    if "town" in field.lower() or "name" in field.lower():
        print(field)

In [ ]:
# Dominant town value QA

missing_town_count = 0

with arcpy.da.SearchCursor(subbasin_polygons, ["GRIDCODE", "Dominant_Town"]) as cursor:
    for gridcode, dominant_town in cursor:
        if dominant_town in [None, ""]:
            missing_town_count += 1
            print(f"Missing dominant town for subbasin: {gridcode}")

print("Dominant town QA:")
print(f"Missing dominant towns : {missing_town_count}")
print(f"Town QA                : {'PASS' if missing_town_count == 0 else 'REVIEW'}")

# TASK 8: FINAL NOTEBOOK 1 HANDOFF CHECK

#### Step 1: Confirm Outputs Needed by Notebook 2

In [ ]:
# Define Notebook 1 outputs required by Notebook 2

notebook_1_outputs = {
    "Study Area": study_area,
    "Hydro DEM": dem_hydro,
    "Flow Direction": flow_dir,
    "Flow Accumulation": flow_acc,
    "Stream Links": stream_links,
    "Stream Network": stream_network,
    "Subbasin Raster": subbasin_raster,
    "Subbasin Polygons": subbasin_polygons,
    "Projected Soils": soils_projected,
    "Projected LULC": lulc_projected}

missing_outputs = []

print("Notebook 1 Output QA")
print("-" * 50)

for name, dataset in notebook_1_outputs.items():
    exists = arcpy.Exists(dataset)
    print(f"{name:<25}: {'OK' if exists else 'MISSING'}")

    if not exists:
        missing_outputs.append(name)

outputs_pass = len(missing_outputs) == 0

print()
print(f"Required Output QA: {'PASS' if outputs_pass else 'REVIEW'}")

#### Step 2: Calculate Subbasin Coverage Statistics

In [ ]:
# Calculate HUC12 watershed area

watershed_area_sq_m = 0

with arcpy.da.SearchCursor(study_area, ["SHAPE@AREA"]) as cursor:
    for row in cursor:
        watershed_area_sq_m += row[0]

print("Reference watershed area calculated.")
print(f"Watershed area: {watershed_area_sq_m:,.0f} m²")

In [ ]:
# Calculate subbasin raster area

subbasin_cell_count = 0

with arcpy.da.SearchCursor(subbasin_raster, ["Count"]) as cursor:
    for row in cursor:
        subbasin_cell_count += row[0]

subbasin_raster_area_sq_m = (subbasin_cell_count * swat_cell_size * swat_cell_size)

print("Subbasin raster area calculated.")
print(f"Subbasin raster cells: {subbasin_cell_count:,}")
print(f"Subbasin raster area : {subbasin_raster_area_sq_m:,.0f} m²")

In [ ]:
# Calculate subbasin polygon statistics

polygon_feature_count = int(arcpy.management.GetCount(subbasin_polygons)[0])

subbasin_polygon_area_sq_m = 0
unique_subbasins = set()

with arcpy.da.SearchCursor(subbasin_polygons, ["gridcode", "Subbasin_Area_sq_m"]) as cursor:
    for gridcode, area in cursor:

        unique_subbasins.add(gridcode)
        subbasin_polygon_area_sq_m += area

subbasin_count = len(unique_subbasins)

print("Subbasin polygon statistics calculated.")
print(f"Polygon features      : {polygon_feature_count:,}")
print(f"Unique subbasins      : {subbasin_count:,}")
print(f"Total subbasin area   : {subbasin_polygon_area_sq_m:,.0f} m²")

In [ ]:
# Compare watershed, raster, and polygon coverage

subbasin_raster_coverage = (subbasin_raster_area_sq_m / watershed_area_sq_m * 100)
subbasin_polygon_coverage = (subbasin_polygon_area_sq_m / watershed_area_sq_m * 100)

minimum_subbasin_coverage = 85

subbasin_pass = (subbasin_polygon_coverage >= minimum_subbasin_coverage)

print("Coverage QA")
print("-" * 50)
print(f"Watershed Area      : {watershed_area_sq_m:,.0f} m²")
print(f"Raster Area         : {subbasin_raster_area_sq_m:,.0f} m²")
print(f"Polygon Area        : {subbasin_polygon_area_sq_m:,.0f} m²")
print(f"Raster Coverage     : {subbasin_raster_coverage:.2f}%")
print(f"Polygon Coverage    : {subbasin_polygon_coverage:.2f}%")
print(f"Threshold           : {minimum_subbasin_coverage}%")
print(f"Coverage QA         : {'PASS' if subbasin_pass else 'REVIEW'}")

#### Step 3: Pond / Sink Validation

In [ ]:
# Convert sink groups raster to polygons for pond/sink validation

sink_polygons = os.path.join(temp_gdb, "Sink_Group_Polygons")
inputs_folder = os.path.join(project_folder, "Inputs")
lakes         = os.path.join(inputs_folder, "Lakes_Projected")


if arcpy.Exists(sink_polygons):
    arcpy.management.Delete(sink_polygons)

arcpy.conversion.RasterToPolygon(in_raster=sink_groups, out_polygon_features=sink_polygons, simplify="NO_SIMPLIFY", raster_field="Value")

print("Sink group polygons created for pond/sink validation.")
print(sink_polygons)

In [ ]:
# Compare mapped ponds to DEM sinks

arcpy.management.MakeFeatureLayer(lakes, "lakes_lyr")

total_lakes = int(arcpy.management.GetCount( "lakes_lyr")[0])

arcpy.management.SelectLayerByLocation("lakes_lyr", "INTERSECT", sink_polygons)

matched_lakes = int(arcpy.management.GetCount("lakes_lyr")[0])

lake_coverage = (matched_lakes / total_lakes * 100)

minimum_lake_coverage = 80

pond_pass = (lake_coverage >= minimum_lake_coverage)

print("Pond / Sink Validation QA")
print("-" * 50)
print(f"Mapped Lakes/Ponds : {total_lakes}")
print(f"Matched Lakes      : {matched_lakes}")
print(f"Coverage           : {lake_coverage:.1f}%")
print(f"Threshold          : {minimum_lake_coverage}%")
print(f"Validation QA      : {'PASS' if pond_pass else 'REVIEW'}")

#### Step 4: Final Dashboard

In [ ]:
# Calculate dashboard feature counts

input_stream_count = int(arcpy.management.GetCount(rivers_projected)[0])

town_count = int(arcpy.management.GetCount(towns_projected)[0])

print("Dashboard feature counts calculated.")
print(f"Towns                 : {town_count:,}")
print(f"Input Stream Features : {input_stream_count:,}")

In [ ]:
# Final Notebook 1 Handoff QA Dashboard

ready_for_notebook_2 = (outputs_pass and subbasin_pass)

unrepresented_area_sq_m = (watershed_area_sq_m - subbasin_polygon_area_sq_m)

unrepresented_area_percent = (unrepresented_area_sq_m / watershed_area_sq_m * 100)

print("=" * 65)
print("FINAL NOTEBOOK 1 HANDOFF QA DASHBOARD")
print("=" * 65)

print("WORKFLOW OUTPUTS")
print("-" * 65)
print(f" DEM Resolution             : {swat_cell_size} m")
print(f" Towns                      : {town_count:,}")
print(f" Input Stream Features      : {input_stream_count:,}")
print(f" Generated Stream Segments  : {stream_count:,}")
print(f" Generated Stream Links     : {stream_link_count:,}")
print(f" Unique Subbasins           : {subbasin_count:,}")

print()
print("AREA SUMMARY")
print("-" * 65)
print(f" HUC12 Watershed Area       : {watershed_area_sq_m:,.0f} m²")
print(f" Subbasin Raster Area       : {subbasin_raster_area_sq_m:,.0f} m²")
print(f" Subbasin Polygon Area      : {subbasin_polygon_area_sq_m:,.0f} m²")
print(f" Unrepresented HUC12 Area   : {unrepresented_area_sq_m:,.0f} m²")
print(f" Unrepresented HUC12 Area % : {unrepresented_area_percent:.2f}%")

print()
print("COVERAGE QA")
print("-" * 65)
print(f" Subbasin Raster Coverage   : {subbasin_raster_coverage:.2f}%")
print(f" Subbasin Polygon Coverage  : {subbasin_polygon_coverage:.2f}%")
print(f" Subbasin Coverage QA       : {'PASS' if subbasin_pass else 'REVIEW'}")
print(f" Pond/Sink Validation       : {lake_coverage:.1f}%")
print(f" Pond/Sink QA               : {'PASS' if pond_pass else 'REVIEW'}")

print()
print("HANDOFF STATUS")
print("-" * 65)
print(f" Required Outputs           : {'PASS' if outputs_pass else 'REVIEW'}")
print(f" Ready for Notebook 2       : {'YES' if ready_for_notebook_2 else 'REVIEW REQUIRED'}")

print("=" * 65)